In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import json
import os
from functools import partial
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import nomad.io.base as loader
import nomad.stop_detection.utils as utils
import nomad.stop_detection.lachesis as LACHESIS
import nomad.stop_detection.sequential as SEQUENTIAL
from nomad.stop_detection.density_based import seqscan_labels
from nomad.stop_detection.dbscan import ta_dbscan_labels
from nomad.stop_detection.density_based import seqscan_labels_per_user
from nomad.stop_detection.lachesis import lachesis_labels_per_user

import nomad.visit_attribution.visit_attribution as visits

from nomad.contact_estimation import compute_stop_detection_metrics
import warnings
warnings.filterwarnings('ignore', message='Input is timezone-aware; assuming UTC')

In [ ]:

def user_maxima_summary(data, algo_prefix, beta_ping, ha, metric="recall", param='delta_roam'):
    data = data.loc[data.algorithm.str.startswith(algo_prefix)]
    stats_by_user = (
        data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param]]
        .reset_index(drop=True)
    )
    e_theta_star = stats_by_user[param].quantile(0.5, interpolation='lower')
    e_x_theta_star = stats_by_user[metric].median()
    lo, hi = stats_by_user[param].quantile([0.025, 0.975])
    in_interval = data.loc[data[param].between(lo, hi)]
    std_by_delta = in_interval.groupby(param)[metric].std()
    mean_metric_by_param = in_interval.groupby(param)[metric].mean()
    return {
        "beta_ping": beta_ping,
        "ha": ha,
        "med_optimal_param": e_theta_star,
        "med_best_metric": round(e_x_theta_star, 4),
        "95pct_param_range": f"[{lo:.1f}, {hi:.1f}]",
        "std_at_optimal_param": std_by_delta.loc[e_theta_star],
        "max_std_in_interval": std_by_delta.max(),
        "min_std_in_interval": std_by_delta.min(),
        "max_mean_performance": mean_metric_by_param.max(),
        "min_interval_performance": mean_metric_by_param.min()
    }

In [ ]:
config_files = ['config_ha13_beta5.json', 'config_ha13_beta6.json',
                'config_ha13_beta7.json', 'config_ha13_beta8.json',
                'config_ha15_beta5.json', 'config_ha15_beta6.json',
                'config_ha15_beta7.json', 'config_ha15_beta8.json',
                'config_ha17_beta5.json', 'config_ha17_beta6.json',
                'config_ha17_beta7.json', 'config_ha17_beta8.json']

POI_LOCATIONS = ['w-x17-y10', 'r-x19-y11']

dist_thresh_values = np.linspace(1, 60, 5)
delta_roam_values  = np.linspace(5, 80, 10)

algos = [*[("seqscan",{"func": seqscan_labels_per_user,
                      "params": {"time_thresh": 45, "dist_thresh": dr, "min_pts": 2,
                                 "back_merge": False, "n_jobs": 2},
                                 "visit_attr": "majority"}) for dr in dist_thresh_values],
        *[("seqscan", {"func": seqscan_labels_per_user,
                       "params": {"time_thresh": 45, "dist_thresh": dr, "min_pts": 3,
                                  "back_merge": False, "n_jobs": 2},
                                  "visit_attr": "majority"}) for dr in dist_thresh_values],
        *[("lachesis", {"func": lachesis_labels_per_user,
                        "params": {"dt_max": 45, "delta_roam": 1.7 * dr, "n_jobs": 2},
                        "visit_attr": "majority"}) for dr in delta_roam_values]]

summarize_stops_with_loc = partial(
    utils.summarize_stop,
    x='x', y='y', timestamp='timestamp',
    keep_col_names=True,
    passthrough_cols=['location', 'user_id', 'cluster'],
    complete_output=False
)

In [ ]:
all_results = []

for config_file in config_files:
    key = config_file.replace('config_', '').replace('.json', '')

    if os.path.exists(f"results_{key}.parquet"):
        print(f"results_{key}.parquet already exists. Skipping save.")
        all_results.append(pd.read_parquet(f"results_{key}.parquet"))
        continue

    with open(config_file, 'r', encoding='utf-8') as f:
        config = json.load(f)

    beta_ping = config['agent_params']['beta_ping']
    ha = config['agent_params']['ha'] * 15

    poi_table  = gpd.read_parquet(config["buildings_file"]).rename(columns={"id": "location"})
    sparse_df  = loader.from_file(config["output_files"]["sparse_path"], format="parquet")
    sparse_df.drop(columns=['datetime'], inplace=True)
    diaries_df = loader.from_file(config["output_files"]["diaries_path"], format="parquet").rename(
        columns={"identifier": "user_id"})

    poi_subset = poi_table.loc[poi_table.location.isin(POI_LOCATIONS)]

    sparse_df['precomp_locations'] = visits.poi_map(
        sparse_df, poi_table=poi_subset, data_crs='EPSG:3857',
        max_distance=20, location_id='location', x='x', y='y'
    )

    results_list = []
    for algo_name, algo_config in tqdm(algos, desc=key):
        labels = algo_config["func"](sparse_df, **algo_config["params"],
                                     x="x", y="y", timestamp='timestamp',
                                     user_id='user_id')
        sparse_df['cluster']  = labels
        sparse_df['location'] = sparse_df['precomp_locations']

        sparse_df['location'] = visits.point_in_polygon(
            sparse_df, poi_table=poi_subset, data_crs='EPSG:3857',
            max_distance=20, location_id='location',
            method=algo_config["visit_attr"], x='x', y='y',
            recompute_location=False
        )

        stops_all = (
            sparse_df[sparse_df.cluster != -1]
            .groupby(['user_id', 'cluster'], as_index=False)
            .apply(summarize_stops_with_loc, include_groups=False).reset_index()
        )

        for user in diaries_df.user_id.unique():
            stops = stops_all[stops_all.user_id == user]
            truth = diaries_df.query("user_id==@user")

            metrics = compute_stop_detection_metrics(
                stops=stops, truth=truth, user_id=user, algorithm=algo_name,
                traj_cols={'location_id': 'location'}, timestamp='timestamp'
            )
            metrics['key']        = key
            metrics['beta_ping']  = beta_ping
            metrics['ha']         = ha
            metrics['dist_thresh'] = algo_config["params"].get("dist_thresh", np.nan)
            metrics['delta_roam']  = algo_config["params"].get("delta_roam", np.nan)
            metrics['min_pts'] = algo_config["params"].get("min_pts", np.nan)
            results_list.append(metrics)

        sparse_df.drop(columns=['cluster', 'location'], inplace=True)

    config_df = pd.DataFrame(results_list)
    config_df.to_parquet(f"results_{key}.parquet", index=False)
    all_results.append(config_df)
    print(f"  {len(config_df)} rows saved to results_{key}.parquet")

results_df = pd.concat(all_results, ignore_index=True)
results_df.to_parquet("results_all_configs.parquet", index=False)
print(f"\nTotal rows: {len(results_df)}")

In [ ]:
def stats_summary(data, metric="recall", param='delta_roam', param2=None):
    data = data.dropna(subset=[param])
    
    if param2:
        stats_by_user = (data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param, param2, "ha", "beta_ping"]].reset_index(drop=True))
        param2_star = stats_by_user[param2].value_counts().idxmax()
        theta_star = stats_by_user[stats_by_user[param2] == param2_star][param].quantile(0.5, interpolation='lower')
        mean_acc_theta_star = data.loc[data[param]==theta_star][metric].mean()
        std_acc_theta_star = data.loc[data[param]==theta_star][metric].std()
        
    else:
        stats_by_user = (data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param, "ha", "beta_ping"]].reset_index(drop=True))
        theta_star = stats_by_user[param].quantile(0.5, interpolation='lower')
        mean_acc_theta_star = data.loc[data[param]==theta_star][metric].mean()
        std_acc_theta_star = data.loc[data[param]==theta_star][metric].std()

    return pd.DataFrame({"algorithm": data['algorithm'].iloc[0],
                         "ha": stats_by_user["ha"].iloc[0],
                         "beta_ping": stats_by_user["beta_ping"].iloc[0],
                         "mean_acc_theta_star": mean_acc_theta_star,
                         "mean_acc_theta_star_i": round(stats_by_user[metric].mean(), 4), # oracle performance
                         "std_acc_theta_star": std_acc_theta_star,
                         "theta_star": theta_star,
                         'min_pts': param2_star if param2 else np.nan},
                         index=[param])

In [ ]:
results_df

In [ ]:
results1 = results_df.groupby("key").apply(stats_summary, metric="recall", param='delta_roam').reset_index().drop(columns=['key', 'level_1'])
results2 = results_df.groupby("key").apply(stats_summary, metric="recall", param='dist_thresh', param2='min_pts').reset_index().drop(columns=['key', 'level_1'])

In [ ]:
results1

In [ ]:
results2

In [ ]:
pd.concat([results1, results2], ignore_index=True).sort_values(['beta_ping', 'ha'])